# Field mapping

Projecting points $p$ from surface tangent space to a (quadratic bezier) curve $S$ and building SDF-base field.

$$
p_{uv} \to p^* \in S
$$

Assuming the curve is flat enough to calculate everything in 2D.

Some elevation characteristics are still useful to calculate bumping and overlaying.

#### Closest point

- $p$ — sample point in UV space
- $p^* = S(t^*)$ — nearest point on curve

$$
t^* = \argmin_t{||p - S(t)||^2}
$$

#### General solution

- $S(t)$ — general parametric curve
- $F(t) = S'(t)$ ­— first derivative, velocity/flow
- $v(t) = S(t) - p$ — vector point to curve

$$
(p - S(t))·F(t) = 0
\tag{perpendicularity}
$$

$$
\frac{dv^2}{dt}(t) = 0
\tag{min disquance}
$$

The equations are equivalent and almost identical.

Solving for $t$ gives points with locally minimum distance.
Globally minimum should be chosen by additional computations.

#### Distance

- $v_p = p - p^*$ — vector from a projection point to the sample

$$D^2 = ||v_p||^2 = v_p·v_p$$

Doesn't make sense to squaeroot because it can be used as is.

#### Orientation

- $v = p - p^*$ — vector from a projection point to the sample
- $f = S'(t^*)$ ­— tangential flow at the projection point
- $z = (v × f)_z = v_x f_y - v_y f_x$ — z-component of cross product of 2D vectors into 3D

$$
\begin{cases}
z > 0 \implies \text{at right-hand side} \\
z < 0 \implies \text{at left-hand side} \\
z = 0 \implies \text{exact on the curve, should not happen}\\
\end{cases}
$$

Numeric value $z = |v||f|\sin\phi$


In [1]:
from typing import NamedTuple, Self

import math
import numpy as np
import numpy.random as nprand
from numpy.typing import NDArray

Vec = np.ndarray[tuple[int], np.dtype[np.floating]]  # 2D or 3D
vec = lambda *args: np.array(args, dtype=float)
asarr = np.asarray
nparr = lambda *args: np.array(args)
npgen = lambda iter: np.fromiter(iter, dtype=float)
dot = lambda v1, v2: float(v1 @ v2)
dots = lambda a1, a2: (a1 * a2).sum(axis=-1)
powers = lambda t: np.array([np.ones(np.shape(t)), t, np.square(t)]).T

In [2]:
from field import Field, save_field

In [3]:
def gridspace(lowbound: float, highbound: float, res: int) -> NDArray:
    """Generate XY sampling space with coordinates in centers of grid cells (pixels)
    Returns array of 2d coords (res×res×2)
    """
    r = (highbound - lowbound) / res / 2  # pixel half-size
    cspace = np.linspace(lowbound, highbound, res, endpoint=False) + r
    X, Y = np.meshgrid(cspace, cspace)
    return np.stack((X, Y)).T

In [4]:
def random_points(num: int, jiggle=1.0):
    """Geneates control points from fixed corners (1, 0) (0, 1) and distorted midpoints"""
    xx = np.linspace(1.0, 0.0, num)
    yy = np.linspace(0.0, 1.0, num)

    jigglet = jiggle / num
    coords = np.stack((xx, yy)).T
    coords[1:-1] += np.random.randn(num - 2, 2) * jigglet
    return coords


def random_points3d(num: int, jiggle=1.0):
    """Geneates control points from fixed corners (1, 0, 0) (0, 1, 0) and distorted midpoints"""
    xx = np.linspace(1.0, 0.0, num)
    yy = np.linspace(0.0, 1.0, num)
    zz = np.zeros(num)

    jigglet = jiggle / num
    coords = np.stack((xx, yy, zz)).T
    coords[1:-1] += np.random.randn(num - 2, 3) * jigglet
    return coords

In [5]:
def disquance(p0: Vec, p1: Vec):
    v = p1 - p0
    return dot(v, v)


def disquances(p0: Vec, pp: NDArray):
    vv = pp - p0
    return (vv * vv).sum(axis=1)


def orientation(frw: Vec, vec: Vec):
    return np.sign(vec[0] * frw[1] - vec[1] * frw[0])


def orientations(frws: NDArray, vecs: NDArray):
    return np.sign(vecs[:, 0] * frws[:, 1] - vecs[:, 1] * frws[:, 0])

## Quadratic Beziel

A segment of a B2-Spline over points is equivalent to B2zier througn midpoints.

- $c_1 = \frac{p_1 + p_2}{2}$
- $c_2 = p_2$
- $c_3 = \frac{p_3 + p_2}{2}$


In [6]:
def split_controls(controls: NDArray) -> tuple[NDArray, NDArray, NDArray]:
    """Split long sequence of spline control points into segments
    Returns: 3 arrays of p1 p2 p3 for each segment"""
    num = controls.shape[0]
    return (
        controls[0 : num - 2],
        controls[1 : num - 1],
        controls[2 : num - 0],
    )


def beziefy_controls(p1: NDArray, p2: NDArray, p3: NDArray) -> tuple[NDArray, NDArray, NDArray]:
    """Convert b2spline control points to b2zier controls
    Returns: c1 c2 c3 (arrays or singular points)"""
    return (p1 + p2) * 0.5, p2, (p2 + p3) * 0.5

### Parametrics

Localized around its apex point at $t=0$

$$
S_{orig}(t) = S_{loc}(t - 0.5) + c_a
$$

- $v_1 = c_1 - c_2$
- $v_3 = c_3 - c_2$
- $v_a = v_3 + v_1$ — parabolic axial vector
- $v_t = v_3 - v_1$ — parabolic tangential vector
- $c_a = c_2 + .25 v_a = S(0)$ — apex point

$$
S(t) - c_a=
\big[1, t, t^2 \big]
\begin{bmatrix}
0 \\
v_t \\
v_a \\
\end{bmatrix}
$$

$$
\frac{dS}{dt}(t) =
\big[1, t, t^2 \big]
\begin{bmatrix}
v_t \\
2 v_a \\
0 \\
\end{bmatrix}
$$


Possible special cases

- $v_1 = 0, v_3 = 0$ all collapsed to single point $c_2$
- $v_1 = 0, v_3 \ne 0 \implies v_a = v_3, v_t = v_3$ still valid
- $v_1 \ne 0, v_3 = 0 \implies v_a = v_1, v_t = -v_1$ still valid
- $v_1 + v_3 = 0 \implies v_a = 0, v_t = -2 v_3, c_a = c_2$ linear case $S(t) = c_2 - 2 v_3 t$
- $v_1 = v_3 \implies v_t = 0, v_a = 2 v_3$ collapsed parabola, $S(t) = c_a + 2 v_3 t^2$ still valid


v_a = (c_3 - c_2) + (c_1 - c_2) = c_3 + c_1 - 2 \* c_2
v_t = (c_3 - c-2) - (c_1 - c_2) = c_3 - c_1


In [7]:
class Params(NamedTuple):
    ca: Vec | NDArray
    vt: Vec | NDArray
    va: Vec | NDArray

    @property
    def shape(self):
        return self.ca.shape[:-1]

    @property
    def dim(self):
        return self.ca.shape[-1]


# works both for singular and array
def beziel_params(c1: Vec | NDArray, c2: Vec | NDArray, c3: Vec | NDArray) -> Params:
    """Calculate parabolic basis ca vt va"""
    va = c3 + c1 - 2 * c2
    vt = c3 - c1
    ca = c2 + 0.25 * va
    return Params(ca, vt, va)


# for singular params -> matrix
def beziel_M(params: Params):
    return np.array([np.zeros(params.dim), params.vt, params.va])


# for singular params -> matrix
def beziel_Mdt(params: Params):
    return np.array([params.vt, 2 * params.va, np.zeros(params.dim)])


# # for arrayed params -> array[matrix]
def bezielm_M(params: Params) -> NDArray:
    buff = np.zeros((*params.shape, 3, params.dim))
    buff[:, 1] = params.vt
    buff[:, 2] = params.va
    return buff


# for arrayed params -> array[matrix]
def bezielm_Mdt(params: Params) -> NDArray:
    buff = np.zeros((*params.shape, 3, params.dim))
    buff[:, 0] = params.vt
    buff[:, 1] = 2 * params.va
    return buff

In [8]:
# works for singular params and both singular and arrayed t


def beziel_curve_(params: Params, t: NDArray):
    return powers(t) @ beziel_M(params)


def beziel_curve(params: Params, t: NDArray):
    return beziel_curve_(params, t - 0.5) + params.ca


def beziel_flow_(params: Params, t: NDArray):
    return powers(t - 0.5) @ beziel_Mdt(params)


def beziel_flow(params: Params, t: NDArray):
    return beziel_flow_(params, t - 0.5)

---


In [9]:
# from widgets import Curve3DView

In [10]:
# view = Curve3DView(beziel_curve, beziel_flow)
# display(view)

In [11]:
# points = random_points3d(3)
# params = beziel_params(*points)

# view.controls = points
# view.params = params

---


# Projecting

- $p_a = p - c_a$ — point relative to apex
- $v_p(t)|p = p - S(t)$ ­— vector from curve to sample point
- $v_p^2 = v_p·v_p = |v_p|^2$ — squared distance from curve to point

Equation for local minima/maxima:

$$
\frac{dv_p^2}{dt} =
[1, t, t^2, t^3]
\begin{pmatrix}
-p_a v_t \\
-2 p_a v_a + v_t^2 \\
3 v_a v_t \\
2 v_a^2 \\
\end{pmatrix}
= 0
$$

There are 1 up to 3 bending points. Need to select the best.


Edge cases:

- $v_a = 0, v_t = 0$ all collapsed to one point
    - $t = 0$
- $v_a = 0, v_t \ne 0$ collapsed to line, still valid
    - $v_t^2 t - p_a v_t = 0$
    - $t = \frac{p_a v_t}{v_t^2}$
- $v_a \ne 0, v_t = 0$ squized to line
    - $-2p_a v_a t + 2 v_a^2 t^2 = 0$
    - $t_0 = 0$
    - $-2p_a v_a + 2 v_a^2 t = 0$
    - $t = \frac{p_a v_a}{v_a^2}$


In [12]:
from fqs import single_cubic, single_quadratic  # fast

realroots = lambda roots: {r.real for r in roots if r.imag == 0}

$$a_3 t^3 + a_2 t^2 + a_1 t + a_0 = 0$$

Special cases

- $a_3 t^3 + a_0 = 0$ // $a_1 = a_2 = 0$
    - $t = \sqrt[3]{-a_0/a_3}$
- $a_3 t^3 = 0$ // $a_2 = a_1 = a_0 = 0$
    - $t=0$
- $a_3 t^3 + a_1 t = 0$ // $a_2 = a_0 = 0$
    - $t_0 = 0$
    - $a_3 t^2 + a_1 = 0$
    - $a_1/a_3 \lt 0$
        - $t = \sqrt{-a_1/a_3}$
- $a_3 t^3 + a_2 t^2 = 0$ // $a_1 = a_0 = 0$
    - $t_0 = 0$
    - $a_3 t + a_2 = 0$
    - $t = -a_2/a_3$
- $a_3 t^3 + a_2 t^2 + a_1 t = 0$ // $a_0 = 0$
    - $t_0 = 0$
    - $a_3 t^2 + a_2 t + a_1 = 0$
    - quadratic subcase


In [13]:
def solve_equation(params: Params, sample: Vec) -> set[float]:
    """Solve the equation of perpendicularity = extrema of square distance
    Returns set of uniq t (in local basis -0.5)
    Range not checked
    """
    ca, vt, va = params
    pa = ca - sample  # -pa

    dot_aa = dot(va, va)
    dot_at = dot(va, vt)
    dot_tt = dot(vt, vt)
    dot_pa = dot(pa, va)
    dot_pt = dot(pa, vt)

    za, zt = np.isclose([dot_aa, dot_tt], 0.0)

    if za and zt:
        return {0.0}
    elif za:
        return {dot_pt / dot_tt}
    elif zt:
        return {0.0, dot_pa / dot_aa}

    a0 = dot_pt
    a1 = 2 * dot_pa + dot_tt
    a2 = 3 * dot_at
    a3 = 2 * dot_aa

    z0, z1, z2 = np.isclose([a0, a1, a2], 0.0)

    if z0:
        if z1 and z2:
            return {0.0}
        elif z1:
            return {0.0, -a2 / a3}
        elif z2:
            roots = {0.0}
            a13 = a1 / a3
            if a13 < 0:
                roots.add(math.sqrt(-a13))
            return roots
        else:
            roots = {0.0}
            roots |= realroots(single_quadratic(a3, a2, a1))
            return roots
    elif z1 and z2:
        return {math.cbrt(-a0 / a3)}
    else:
        return realroots(single_cubic(a3, a2, a1, a0))

In [14]:
def projection_all(params: Params, sample: Vec) -> tuple[NDArray, NDArray, NDArray, NDArray, NDArray]:
    """Find all potential projections (up to 3) from sample to curve"""

    tt = solve_equation(params, sample)
    tt = npgen(tt) + 0.5

    pnts = beziel_curve(params, tt)
    vecs = sample - pnts
    dsqs = dots(vecs, vecs)
    flws = beziel_flow(params, tt)
    sgns = orientations(flws, vecs)

    return tt, pnts, dsqs, flws, sgns

In [15]:
def projection(params: Params, sample: Vec) -> tuple[float, Vec, float, Vec, float]:
    """Project a single point to curve and get all the characteristics"""

    tt = solve_equation(params, sample)
    tt = npgen(tt) + 0.5

    pnts = beziel_curve(params, tt)
    vecs = sample - pnts
    dsqs = dots(vecs, vecs)

    best = np.argmin(dsqs)
    arc = tt[best]
    pnt = pnts[best]
    vec = vecs[best]
    dsq = dsqs[best]
    flw = beziel_flow(params, arc)
    sgn = orientation(flw, vec)

    return arc, pnt, dsq, flw, sgn


# vectorized over array of sample points
np_projection = np.vectorize(projection, signature="(d) -> (),(d),(),(d),()", excluded={0})

In [16]:
def pickmap(params: Params, sample: Vec) -> Field:
    arc, pnt, dsq, flw, sgn = projection_all(params, sample)
    samples = np.full((*arc.shape, sample.shape[0]), sample)
    return Field(samples, arc, pnt, dsq, flw, sgn)


def pointmap(params: Params, sample: Vec) -> Field:
    arc, pnt, dsq, flw, sgn = projection(params, sample)
    return Field(nparr(sample), nparr(arc), nparr(pnt), nparr(dsq), nparr(flw), nparr(sgn))


def fieldmap(params: Params, samples: NDArray) -> Field:
    arc, pnt, dsq, flw, sgn = np_projection(params, samples)
    return Field(samples, arc, pnt, dsq, flw, sgn)

---

# Exploring


In [17]:
from ipywidgets import widgets as wg
from plotly import graph_objects as go
from plotly import colors

from widgets import FieldView

### Single Bezier


In [18]:
view = FieldView()
display(view)

FieldView(children=(FigureWidget({
    'data': [{'hoverinfo': 'skip',
              'marker': {'color': 'black…

In [23]:
points = random_points3d(3)
params3d = beziel_params(*points)
params2d = beziel_params(*points[:, :2])

tspace = np.linspace(0.0, 1.0, 32)
gspace = gridspace(0.0, 1.0, 32)

curve = beziel_curve(params3d, tspace)
field = fieldmap(params2d, gspace)

view.controls = points
view.curve = curve
view.field = field

In [20]:
view.picking = lambda sample: pickmap(params2d, sample)